## 1. Загрузка данных (`train.csv`)



In [1]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

csv_path = Path('train.csv')
if not csv_path.exists():
    raise FileNotFoundError(
        "Файл train.csv не найден. Загрузите его рядом с hw_3.ipynb "
        "(в Google Colab — в папку /content)."
    )

df = pd.read_csv(csv_path)
print('Размер таблицы (строки, столбцы):', df.shape)
print('Первые 5 записей:')
display(df.head())

Размер таблицы (строки, столбцы): (891, 12)
Первые 5 записей:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Основная информация о датасете



In [2]:
print('Информация о структуре данных:')
df.info()

print('\nТипы данных:')
display(df.dtypes.rename('Тип данных').to_frame())

missing = pd.DataFrame({
    'Пропусков': df.isna().sum(),
    'Доля пропусков, %': (df.isna().mean() * 100).round(2),
})
print('\nПропущенные значения:')
display(missing)

print('\nЧисловые столбцы — среднее, медиана, минимум, максимум и др.:')
display(df.describe().round(2))

print('\nКатегориальные столбцы — число уникальных значений и популярные значения:')
display(df.describe(include=['object']))

Информация о структуре данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Типы данных:


,Тип данных
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64



Пропущенные значения:


,Пропусков,"Доля пропусков, %"
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00



Числовые столбцы — среднее, медиана, минимум, максимум и др.:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33



Категориальные столбцы — число уникальных значений и популярные значения:


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,1601,G6,S
freq,1,577,7,4,644


## 3. Доля выживших по классу (`Pclass`)


In [3]:
survival_by_class = (
    df.groupby('Pclass')['Survived']
      .agg(Пассажиров='count', Выживших='sum', Доля_выживших='mean')
      .reset_index()
)
survival_by_class['Выживаемость, %'] = (
    survival_by_class['Доля_выживших'] * 100
).round(2)
survival_by_class = survival_by_class.drop(columns='Доля_выживших')
display(survival_by_class)

,Pclass,Пассажиров,Выживших,"Выживаемость, %"
0,1,216,136,62.96
1,2,184,87,47.28
2,3,491,119,24.24


## 4. Самые популярные мужское и женское имена


In [4]:
def get_first_name(full_name):
    # Отделяем фамилию и титул от остальной части строки.
    after_comma = str(full_name).split(',', 1)[-1].strip()
    title_and_name = after_comma.split('.', 1)
    title = title_and_name[0].strip()
    remainder = title_and_name[1].strip() if len(title_and_name) > 1 else after_comma

    # У замужних женщин первое слово может быть именем мужа.
    if title in ('Mrs', 'Mme'):
        parentheses = re.search(r'\(([^)]+)\)', remainder)
        if parentheses:
            remainder = parentheses.group(1)

    # Извлекаем первое имя, оставляя латинские буквы и дефис.
    match = re.search(r'[A-Za-zÀ-ÿ]+(?:-[A-Za-zÀ-ÿ]+)*', remainder)
    return match.group(0) if match else None


df['FirstName'] = df['Name'].apply(get_first_name)

sex_names = {'male': 'Мужское', 'female': 'Женское'}
for sex, label in sex_names.items():
    popularity = df.loc[df['Sex'] == sex, 'FirstName'].value_counts()
    leaders = popularity[popularity == popularity.max()]
    print(f'{label} имя — наиболее популярное:')
    display(leaders.rename('Количество пассажиров').to_frame())

Мужское имя — наиболее популярное:


,Количество пассажиров
FirstName,
William,35


Женское имя — наиболее популярное:


,Количество пассажиров
FirstName,
Anna,15


## 5. Самые популярные имена мужчин и женщин в каждом классе


In [5]:
rows = []
for pclass in sorted(df['Pclass'].dropna().unique()):
    for sex, label in sex_names.items():
        counts = df.loc[
            (df['Pclass'] == pclass) & (df['Sex'] == sex), 'FirstName'
        ].value_counts()
        if counts.empty:
            continue
        leaders = counts[counts == counts.max()]
        for first_name, count in leaders.items():
            rows.append({
                'Класс': int(pclass),
                'Пол': label,
                'Популярное имя': first_name,
                'Количество': int(count),
            })

most_popular_by_class = pd.DataFrame(rows)
display(most_popular_by_class)

,Класс,Пол,Популярное имя,Количество
0,1,Мужское,William,11
1,1,Женское,Margaret,5
2,1,Женское,Elizabeth,5
3,2,Мужское,William,9
4,2,Женское,Elizabeth,5
5,3,Мужское,William,15
6,3,Женское,Anna,9


## 6. Пассажиры старше 44 лет


In [6]:
older_than_44 = df.loc[df['Age'] > 44]
print('Число пассажиров старше 44 лет:', len(older_than_44))
display(older_than_44)

Число пассажиров старше 44 лет: 115


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,858,1,1,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S,Peter
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S,Margaret
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S,Sallie
873,874,0,3,"Vander Cruyssen, Mr. Victor",male,47.0,0,0,345765,9.0000,NaN,S,Victor


## 7. Мужчины младше 44 лет


In [7]:
men_under_44 = df.loc[(df['Age'] < 44) & (df['Sex'] == 'male')]
print('Число мужчин младше 44 лет:', len(men_under_44))
display(men_under_44)

Число мужчин младше 44 лет: 368


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.500,NaN,S,Frederick
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S,Henry
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000,NaN,S,Juozas
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000,C148,C,Karl


## 8. Количество n-местных кабин (с 2, 3, 4 ... пассажирами)


In [8]:
# Не учитываем пассажиров с неизвестным Cabin.
known_cabins = df.dropna(subset=['Cabin'])

# Сколько пассажиров записано в каждом значении Cabin.
passengers_per_cabin = known_cabins.groupby('Cabin')['PassengerId'].nunique()

# Сколько значений Cabin встречается с n пассажирами.
cabins_by_occupancy = (
    passengers_per_cabin.value_counts()
    .sort_index()
    .rename_axis('Человек в группе')
    .rename('Число записей Cabin')
    .reset_index()
)

print('Заполнено поле Cabin:', len(known_cabins), 'из', len(df))
print('Группы с двумя и более пассажирами:')
display(cabins_by_occupancy.loc[cabins_by_occupancy['Человек в группе'] >= 2])

print('\nВсе размеры групп (включая одиночные):')
display(cabins_by_occupancy)

Заполнено поле Cabin: 204 из 891
Группы с двумя и более пассажирами:


,Человек в группе,Число записей Cabin
1,2,38
2,3,5
3,4,3



Все размеры групп (включая одиночные):


,Человек в группе,Число записей Cabin
0,1,101
1,2,38
2,3,5
3,4,3
